In [4]:
import torch
import lightning.pytorch as pl

In [5]:
import time

class Timer():
    def __init__(self):
        pass
    
    def __enter__(self):
        self.start_time = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(time.time() - self.start_time)

In [6]:
# x = torch.randn((3, 3))
x = torch.randint(-15, 15, (3,3))
print(x)

tensor([[-10,  -6, -11],
        [ 13, -12,  -6],
        [ -5,  -7,   1]])


## BReLU 기본형

In [7]:
class BReLU(pl.LightningModule):
    def __init__(self):
        super(BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        return x * epsilon.to(self.device)

In [8]:
brelu = BReLU()
brelu(x)

tensor([[-0., -0., -0.],
        [13., -0., -0.],
        [-0., -0.,  0.]])

In [9]:
print(brelu)

BReLU()


## Leaky BReLU

In [10]:
class Leaky_BReLU(pl.LightningModule):
    def __init__(self):
        super(Leaky_BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = torch.where(epsilon==0, 0.1, 1.0)
        
        return x * epsilon.to(self.device)

In [11]:
leaky = Leaky_BReLU()
leaky(x)

tensor([[-1.0000, -0.6000, -1.1000],
        [13.0000, -1.2000, -0.6000],
        [-0.5000, -0.7000,  1.0000]])

In [12]:
print(leaky)

Leaky_BReLU()


## 점근 속도를 조절하는 BReLU

In [13]:
class VariableBReLU(pl.LightningModule):
    def __init__(self, alpha=1):
        super(VariableBReLU, self).__init__()
        self.alpha = alpha
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=self.alpha * x).sample()
        
        return x * epsilon.to(self.device)
    
    def extra_repr(self) -> str:
        return 'alpha={}'.format(self.alpha)

In [14]:
def mv(**kwargs):
    if 'alpha' in list(kwargs.keys()):
        return VariableBReLU(alpha=kwargs.get('alpha'))

mv(alpha=.1)

VariableBReLU(alpha=0.1)

In [15]:
alpha = 0.1
vb = VariableBReLU(alpha=alpha)
vb(x)

tensor([[-0., -6., -0.],
        [ 0., -0., -0.],
        [-0., -0.,  0.]])

In [16]:
print(vb)

VariableBReLU(alpha=0.1)


## 결과값을 0아니면 1로 만들기

In [17]:
class MeasureAct(pl.LightningModule):
    def __init__(self):
        super(MeasureAct, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = epsilon * (1/(x + 1e-16))
        
        return x * epsilon.to(self.device)

In [18]:
ma = MeasureAct()
ma(x)

tensor([[0., 0., 0.],
        [1., 0., 0.],
        [0., 0., 0.]])

## 일부 원소만 0 아니면 1로 만들기

In [19]:
class SomeMeasureAct(pl.LightningModule):
    def __init__(self):
        super(SomeMeasureAct, self).__init__()
        
    def forward(self, x):
        b = torch.distributions.bernoulli.Bernoulli(logits=x)
        b1 = b.sample()
        b2 = b.sample()

        epsilon = b1*b2*(1/(x+1e-16)) + torch.logical_xor(b1, b2)
        
        return x * epsilon

In [20]:
sma = SomeMeasureAct()
sma(x)

tensor([[-0., -0., -0.],
        [1., -0., -0.],
        [-0., -0., 1.]])

## 확률적으로 ReLU, BReLU를 적용하기

In [21]:
class SARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)
        
    # def extra_repr(self) -> str:
    #     return 'zero={}, one={}'.format(self.zero, self.one)
        

In [22]:
class SARBSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARBSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)

In [23]:
class BatchSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(BatchSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        switch = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, x):
            if b == 0:
                tmp.append(self.zero(batch))
            
            else:
                tmp.append(self.one(batch))
                
        return torch.stack(tmp)

In [24]:
sarb = SARB()
sarb(x)

tensor([[-0., -0., -0.],
        [13., -0., -0.],
        [-0., -0.,  0.]])

In [25]:
sarb

SARB(
  (zero): BReLU()
  (one): ReLU(inplace=True)
)

배치별로 다른 활성화함수 적용 또는 채널별

In [49]:
class BatchWiseSARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=False)):
        super(BatchWiseSARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        switch = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, [x[j] for j in range(x.size(0))]):
            if b == 0:
                # tmp.append(self.zero(batch.clone()))
                tmp.append(self.zero(batch))
            
            else:
                # tmp.append(self.one(batch.clone()))
                tmp.append(self.one(batch))
        return torch.stack(tmp)
    

class BatchWiseSARBSC(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=False)):
        super(BatchWiseSARBSC, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        switch = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, [x[j] for j in range(x.size(0))]):
            if b == 0:
                # tmp.append(self.zero(batch.clone()))
                tmp.append(self.zero(batch))
            
            else:
                # tmp.append(self.one(batch.clone()))
                tmp.append(self.one(batch))
        return torch.stack(tmp)

In [99]:
class ImproveBatchWiseSARBSC(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=False)):
        super(ImproveBatchWiseSARBSC, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=True)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        return self.zero((1-b) * x) + self.one(b * x)

In [90]:
mu = testX.mean(dim=(1,2,3), keepdim=True)
b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()

In [92]:
b.size()

torch.Size([256, 1, 1, 1])

In [94]:
b

tensor([[[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[0.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[0.]]],


        [[[0.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[0.]]],


        [[[0.]]],


        [[[0.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],


        [[[1.]]],




In [96]:
testX * (1-b) + testX * b

tensor([[[[0.9901, 0.2031],
          [0.9563, 0.8200]],

         [[0.1019, 0.0627],
          [0.5041, 0.7682]]],


        [[[0.1769, 0.2531],
          [0.5559, 0.7141]],

         [[0.4249, 0.0318],
          [0.1656, 0.9483]]],


        [[[0.9919, 0.9267],
          [0.9175, 0.4586]],

         [[0.5916, 0.0792],
          [0.6058, 0.9874]]],


        ...,


        [[[0.0396, 0.1902],
          [0.9741, 0.1953]],

         [[0.6636, 0.8595],
          [0.5129, 0.4265]]],


        [[[0.5300, 0.3029],
          [0.7051, 0.3045]],

         [[0.8012, 0.2063],
          [0.5529, 0.0810]]],


        [[[0.9229, 0.0603],
          [0.5688, 0.0109]],

         [[0.8210, 0.8207],
          [0.8945, 0.8044]]]])

In [ ]:
testX = torch.rand((256, 2, 2, 2))
batchsc = BatchSC()

print(testX)
print('after activation')
print(batchsc(testX))

In [73]:
%%timeit -n 100 -r 10
batchwise(testX)

5.4 ms ± 66.5 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [74]:
%%timeit -n 100 -r 10
brelu(testX)

57.4 µs ± 22.6 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [75]:
%%timeit -n 100 -r 10
relu(testX)

6.43 µs ± 1.89 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [100]:
ib = ImproveBatchWiseSARBSC()

In [102]:
%%timeit -n 100 -r 10
ib(testX)

168 µs ± 13.4 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [63]:
relu = torch.nn.ReLU()
with Timer():
    relu(testX)

0.00011444091796875


In [57]:
batchwise = BatchWiseSARBSC()
with Timer():
    batchwise(testX)

0.010975360870361328


In [27]:
tx = torch.randint(-5, 5, (4, 3, 2, 2)).to(torch.float32)
# ty = []
# for i in tx:
#     print(i-1)
#     ty.append(i-1)
# ty = torch.stack(ty)
print(tx)
# print(ty)
mu = tx.mean(dim=(1, 2, 3), keepdim=False)
print(mu.shape)
# print(torch.distributions.bernoulli.Bernoulli(logits=tx.mean(dim=(1,2, 3), keepdim=True)).sample().expand_as(tx))
p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
p = torch.where(p < 0, 0, p)
p = torch.where(p > 1, 1, p)
b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
print(b)
for switch, channel in zip(b, tx):
    print(f'switch = {switch} with channel = {channel}')

tensor([[[[ 0.,  4.],
          [ 3.,  4.]],

         [[ 4., -4.],
          [-1., -3.]],

         [[ 1.,  2.],
          [-4.,  2.]]],


        [[[-1.,  2.],
          [-5.,  0.]],

         [[-5.,  4.],
          [ 4.,  1.]],

         [[-5.,  0.],
          [-4., -4.]]],


        [[[-2., -2.],
          [ 1.,  0.]],

         [[-5., -1.],
          [ 3.,  1.]],

         [[-5., -5.],
          [-3.,  1.]]],


        [[[ 3.,  4.],
          [ 1., -5.]],

         [[-5., -5.],
          [-3., -3.]],

         [[ 4., -4.],
          [-4.,  4.]]]])
torch.Size([4])
tensor([0., 1., 1., 0.])
switch = 0.0 with channel = tensor([[[ 0.,  4.],
         [ 3.,  4.]],

        [[ 4., -4.],
         [-1., -3.]],

        [[ 1.,  2.],
         [-4.,  2.]]])
switch = 1.0 with channel = tensor([[[-1.,  2.],
         [-5.,  0.]],

        [[-5.,  4.],
         [ 4.,  1.]],

        [[-5.,  0.],
         [-4., -4.]]])
switch = 1.0 with channel = tensor([[[-2., -2.],
         [ 1.,  0.]],

        

## 확률 식으로 sin, cos 사용

In [28]:
class BwithSinCos(pl.LightningModule):
    def __init__(self):
        super(BwithSinCos, self).__init__()
        
    def forward(self, x):
        p = -torch.sin(x) * torch.cos(x + (torch.pi/2))
        print('out of 1')
        print((p > 1).nonzero(as_tuple=True))
        print(p[(p > 1).nonzero(as_tuple=True)])
        print('out of 0')
        print((p < 0 ).nonzero(as_tuple=True))
        print(p[(p < 0 ).nonzero(as_tuple=True)])
        print()
        epsilon = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        return x * epsilon.to(self.device)

In [29]:
sc = BwithSinCos()
sc(x)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[ -0.,  -0., -11.],
        [  0.,  -0.,  -0.],
        [ -5.,  -7.,   1.]])

In [30]:
i = torch.tensor([[1.56, torch.pi/2],
                  [0, 1]])

sc(i)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[1.5600, 1.5708],
        [0.0000, 0.0000]])

## 얽힘 (구현 방법 생각 중)

모든 칸을 돌면서 자기 주위의 1 개수를 센 다음 % 2 한 값을 자기 위치에 갖는다.

In [31]:
b = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
o = torch.zeros_like(b)
print(b)
print(o)

tensor([[0., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [32]:
a = torch.randint(0, 2, (3, 4, 5))
a

tensor([[[0, 0, 1, 0, 1],
         [1, 1, 1, 1, 1],
         [0, 0, 0, 0, 1],
         [1, 0, 0, 0, 0]],

        [[0, 1, 1, 1, 0],
         [0, 0, 1, 1, 1],
         [0, 1, 0, 1, 1],
         [1, 1, 1, 1, 1]],

        [[1, 0, 1, 0, 1],
         [1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0],
         [0, 0, 1, 0, 0]]])

In [33]:
(a==1).nonzero(as_tuple=True)
a[(a==1).nonzero(as_tuple=True)] += 1 

In [34]:
(a==1).nonzero(as_tuple=True)

(tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64))

In [35]:
a

tensor([[[0, 0, 2, 0, 2],
         [2, 2, 2, 2, 2],
         [0, 0, 0, 0, 2],
         [2, 0, 0, 0, 0]],

        [[0, 2, 2, 2, 0],
         [0, 0, 2, 2, 2],
         [0, 2, 0, 2, 2],
         [2, 2, 2, 2, 2]],

        [[2, 0, 2, 0, 2],
         [2, 2, 0, 0, 0],
         [2, 2, 0, 2, 0],
         [0, 0, 2, 0, 0]]])

In [36]:
ones = (b==1).nonzero(as_tuple=False)

for i in ones:
    print(i)
    print(b[tuple(i)])

tensor([1, 0])
tensor(1.)
tensor([2, 2])
tensor(1.)


-------------------------------------

In [37]:
f = torch.distributions.bernoulli.Bernoulli(logits=alpha * x)

In [38]:
f.probs

tensor([[0.2689, 0.3543, 0.2497],
        [0.7858, 0.2315, 0.3543],
        [0.3775, 0.3318, 0.5250]])

In [39]:
e = torch.exp(alpha * x) / (torch.exp(alpha * x) + 1)
print(e)

tensor([[0.2689, 0.3543, 0.2497],
        [0.7858, 0.2315, 0.3543],
        [0.3775, 0.3318, 0.5250]])


In [40]:
p = -torch.sin(x) * torch.cos(x+ (torch.pi/2))
print(p)
torch.distributions.bernoulli.Bernoulli(probs=p).sample()

tensor([[0.2960, 0.0781, 1.0000],
        [0.1765, 0.2879, 0.0781],
        [0.9195, 0.4316, 0.7081]])


tensor([[0., 0., 1.],
        [0., 0., 0.],
        [1., 0., 0.]])